# Tutorial 1: Infinite Market Data Generation

## The Problem with Fixed Datasets

Most financial RL research uses historical price data (Yahoo Finance, Quandl, etc.) to train agents. This creates a fundamental limitation:

- **Overfitting**: With only ~20 years of daily data (~5,000 samples), an RL agent memorises the specific sequence rather than learning general market dynamics.
- **Regime blindness**: If your training data only contains a bull market, the agent collapses the first time it encounters a crash.
- **No statistical diversity**: The same autocorrelation, the same volatility clusters, the same tail events — every training run sees identical data.

## Spectral-Env-Core's Solution

Every call to `env.reset()` generates a completely new price path from a calibrated stochastic process. The path is unique — it will never repeat. But it shares the same *statistical DNA* as real markets: fat tails, volatility clustering, regime transitions, autocorrelation.

This means:
- **Infinite training data** — run 10 million episodes and never see the same path twice
- **Automatic regime diversity** — the Markov switching randomly produces bull runs, crashes, and sideways markets
- **Robust agents** — trained on the full distribution of market characters, not a single historical sample

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from spectral_env_core import SpectralTradingEnv

# Configure a single-asset environment
env = SpectralTradingEnv(
    num_assets=1,
    num_steps=252,
    initial_price=100.0,
    volatility=0.25,
    drift=0.08,
    garch_alpha=0.08,
    garch_beta=0.85,
    jump_intensity=2.0,
    jump_mean=-0.02,
    jump_std=0.05,
)

In [ ]:
# Generate 50 unique episodes and plot them
fig, ax = plt.subplots(figsize=(14, 6))

for i in range(50):
    env.reset(seed=i)
    path = env.brownian_path[:, 0]  # shape (253,) for 252 steps
    ax.plot(path, alpha=0.4, lw=0.8)

ax.set_title('50 Unique Market Episodes from the Same Parameters', fontsize=13)
ax.set_xlabel('Trading Day')
ax.set_ylabel('Price ($)')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Each path is generated fresh — none are stored, none repeat.")
print("Your agent trains on the full distribution, not a single historical sample.")

## Why This Matters for RL Training

A PPO agent with `n_steps=2048` and 8 parallel environments consumes ~65 complete episodes per policy update. Over 7 million timesteps, that's ~27,000 unique episodes.

With a fixed dataset of 20 years of daily data, you'd recycle the same 80 episodes (252 days each) hundreds of times. The agent memorises entry points rather than learning dynamics.

With Spectral-Env-Core, those 27,000 episodes are all statistically independent draws from the same calibrated distribution — each with a different regime sequence, different GARCH realisation, different jump events. The agent can't memorise; it must generalise.

In [ ]:
# Demonstrate statistical diversity: collect terminal returns from 1000 episodes
terminal_returns = []

for i in range(1000):
    env.reset(seed=i)
    final_price = env.brownian_path[-1, 0]
    init_price  = env.brownian_path[0, 0]
    terminal_returns.append((final_price / init_price - 1) * 100)

fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(terminal_returns, bins=60, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0, color='black', lw=1, ls='--')
ax.axvline(np.mean(terminal_returns), color='red', lw=2, label=f'Mean: {np.mean(terminal_returns):.1f}%')
ax.set_title('Distribution of 1-Year Returns (1000 episodes)', fontsize=13)
ax.set_xlabel('Return (%)')
ax.set_ylabel('Frequency')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Return statistics:")
print(f"  Mean:   {np.mean(terminal_returns):+.1f}%")
print(f"  Std:    {np.std(terminal_returns):.1f}%")
print(f"  Min:    {np.min(terminal_returns):+.1f}%")
print(f"  Max:    {np.max(terminal_returns):+.1f}%")
print(f"  Skew:   {float(np.mean(((np.array(terminal_returns) - np.mean(terminal_returns)) / np.std(terminal_returns))**3)):.2f}")

Notice the negative skew — that's the jump diffusion (mean jump size is negative) creating realistic crash tail events. An agent trained on this distribution has already "seen" crashes before they happen in production.

---

**Next tutorial:** [02 — Market Realism](02_market_realism.ipynb) — how the stochastic layers combine to produce statistically accurate price dynamics.